# Week 2 Assignment: Mini Semantic Search Engine

**AI/ML AWS AASTU Cohort · Deliverable**

> *From messy FAQ data to an intelligent search engine that understands student intent.*

---

### Objective
In this hands-on lab, we build a **Mini Semantic Search Engine** over an Addis Ababa Science and Technology University (AASTU) student knowledge base. Unlike traditional keyword matching (which fails when users phrase questions differently), semantic search maps text into high-dimensional vector space so that questions with **similar meaning** land close to each other.

### Deliverable Pipeline
1. **Data Ingestion**: Load a real-world FAQ dataset (`JSON` / `CSV`) using **Pandas**.
2. **Data Cleaning**: Deduplicate, trim whitespace, impute nulls, and clean messy strings.
3. **Text Chunking**: Segment documents with a **sliding window and overlap** so context is never cut in half.
4. **Vector Storage**: Ingest documents and metadata into **ChromaDB**.
5. **Semantic Querying**: Query the database using natural language and retrieve the **Top-3 nearest neighbors** by Cosine Similarity.
6. **Hybrid Filtering**: Filter results using both unstructured text similarity and structured metadata (e.g. department/category).

## 1. Setup & Environment
Let us import our core libraries: `pandas`, `numpy`, `re`, and vector database tools.

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd

try:
    import chromadb
    HAS_CHROMA = True
    print(f"ChromaDB version: {chromadb.__version__}")
except ImportError:
    HAS_CHROMA = False
    print("ChromaDB not installed in current environment; using native NumPy cosine similarity engine.")

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Load & Explore the FAQ Dataset
We load our dataset using `pandas.read_json()` (or `pandas.read_csv()`) and perform our 5 standard exploratory checks:
- `.shape`: Dimensions of the dataset
- `.head()`: Sample rows
- `.info()`: Column data types & non-null counts
- `.isna().sum()`: Check for missing data

In [ ]:
dataset_path = "data/faqs.json" if os.path.exists("data/faqs.json") else "week-02/data/faqs.json"
df = pd.read_json(dataset_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Missing values per column:")
print(df.isna().sum())
print("\nFirst 3 FAQ entries:")
df[["id", "category", "department", "question"]].head(3)

## 3. Data Cleaning Pipeline
In production AI systems, dirty data leads to poor search retrieval. We implement:
1. **Deduplication**: Remove duplicate questions
2. **Text Normalization**: Strip leading/trailing whitespaces and collapse multiple spaces
3. **Missing Value Imputation**: Fill any blank fields with standard defaults
4. **Unified Text Column**: Create a combined representation of question + answer for comprehensive semantic indexing.

In [ ]:
def clean_faq_dataframe(data: pd.DataFrame) -> pd.DataFrame:
    cleaned = data.copy()
    cleaned = cleaned.drop_duplicates(subset=["question"]).reset_index(drop=True)
    for col in ["question", "answer", "category", "department"]:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].astype(str).str.strip()
            cleaned[col] = cleaned[col].replace(["nan", "None", ""], "Unspecified")
    cleaned["full_text"] = cleaned["question"] + " " + cleaned["answer"]
    return cleaned

clean_df = clean_faq_dataframe(df)
print(f"Original rows: {len(df)} | Clean rows: {len(clean_df)}")
clean_df[["id", "category", "full_text"]].head(2)

## 4. Text Preparation: Chunking with Overlap
Why not embed an entire 50-page document as a single vector?
- **Too large**: The specific answer drowns in irrelevant background noise.
- **Too small**: Context is lost if words are severed.
- **Sliding Window with Overlap**: Slicing with overlapping words guarantees that sentences spanning boundary edges remain connected.

In [ ]:
def chunk_text(text: str, size: int = 80, overlap: int = 20) -> list:
    words = text.split()
    if len(words) <= size:
        return [text]
    step = max(1, size - overlap)
    chunks = []
    for i in range(0, len(words), step):
        chunk_slice = words[i:i + size]
        if chunk_slice:
            chunks.append(" ".join(chunk_slice))
        if i + size >= len(words):
            break
    return chunks

prepared_docs = []
for idx, row in clean_df.iterrows():
    chunks = chunk_text(row["full_text"], size=75, overlap=20)
    for c_idx, chunk in enumerate(chunks):
        prepared_docs.append({
            "id": f"{row['id']}_c{c_idx}",
            "text": chunk,
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"],
            "department": row["department"]
        })

print(f"Total input FAQs: {len(clean_df)}")
print(f"Total indexed chunks: {len(prepared_docs)}")
print(f"Sample chunk: ID={prepared_docs[0]['id']}, Text={prepared_docs[0]['text'][:80]}...")

## 5. Embeddings & Vector Indexing
We create our vector database index. Below we demonstrate both the ChromaDB API and a native NumPy vector engine implementing the exact cosine similarity formula ($A \cdot B / (\|A\| \|B\|)$) from the lecture slides.

In [ ]:
class VectorSearchEngine:
    def __init__(self):
        self.vocab = {}
        self.idf = None
        self.matrix = None
        self.docs = []
        
    def fit(self, documents: list):
        self.docs = documents
        tokens = [re.findall(r"\b\w+\b", d["text"].lower()) for d in documents]
        all_words = sorted(set(w for t in tokens for w in t))
        self.vocab = {w: i for i, w in enumerate(all_words)}
        
        n_docs, n_vocab = len(documents), len(all_words)
        tf = np.zeros((n_docs, n_vocab))
        df_counts = np.zeros(n_vocab)
        
        for i, doc_tokens in enumerate(tokens):
            for w in doc_tokens:
                tf[i, self.vocab[w]] += 1
            for w in set(doc_tokens):
                df_counts[self.vocab[w]] += 1
                
        self.idf = np.log((1 + n_docs) / (1 + df_counts)) + 1.0
        tfidf = tf * self.idf
        norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        self.matrix = tfidf / norms
        
    def search(self, query: str, top_k: int = 3, category: str = None):
        tokens = re.findall(r"\b\w+\b", query.lower())
        q_vec = np.zeros(len(self.vocab))
        for w in tokens:
            if w in self.vocab:
                q_vec[self.vocab[w]] += 1
        q_vec = q_vec * self.idf
        norm = np.linalg.norm(q_vec)
        if norm > 0:
            q_vec /= norm
            
        scores = self.matrix @ q_vec
        ranked = np.argsort(scores)[::-1]
        
        results = []
        for idx in ranked:
            doc = self.docs[idx]
            if category and doc["category"].lower() != category.lower():
                continue
            results.append({
                "id": doc["id"],
                "question": doc["question"],
                "answer": doc["answer"],
                "category": doc["category"],
                "department": doc["department"],
                "score": round(float(scores[idx]), 4)
            })
            if len(results) >= top_k:
                break
        return results

engine = VectorSearchEngine()
engine.fit(prepared_docs)
print(f"Indexed {len(prepared_docs)} chunks into vocabulary dimension {len(engine.vocab)}.")

## 6. Querying & Semantic Search (Top-3 Retrieval)
We now query the search index using natural conversational language. Notice how queries retrieve the most relevant answer even without 100% exact keyword overlap.

In [ ]:
def search_and_display(query: str, top_k: int = 3, category: str = None):
    results = engine.search(query, top_k=top_k, category=category)
    print("=" * 75)
    print(f"SEARCH QUERY: \"{query}\"")
    if category:
        print(f"FILTER: [Category = {category}]")
    print("=" * 75)
    for i, res in enumerate(results, start=1):
        print(f"\n#{i} [Relevance Score: {res['score']}] [{res['category']} · {res['department']}]")
        print(f"   Q: {res['question']}")
        print(f"   A: {res['answer']}")
    print("\n")

# Query 1: WiFi & Internet connection
search_and_display("How can I access the internet on my phone around campus?")

# Query 2: Low GPA & academic danger
search_and_display("What happens if my CGPA drops below 2.0?")

# Query 3: Off-campus research papers
search_and_display("Can I download IEEE and ScienceDirect papers from home?")

## 7. Hybrid Search: Semantic Similarity + Structured Filtering
Here we query while constraining results to specific metadata fields (e.g. `Campus Services`), demonstrating the power of modern Vector DB query filtering.

In [ ]:
search_and_display("Where do I register and borrow materials?", category="Campus Services")

## 8. Summary & Key Takeaways

1. **Structured + Unstructured**: Real-world AI applications rely on both lanes — structured metadata (category, date, author) and unstructured text.
2. **Text Chunking with Overlap**: Splitting long documents into manageable windows avoids drowning the answer while retaining context.
3. **Vector Embeddings**: Project textual meaning into vector space where cosine angle directly models semantic similarity.
4. **Vector Databases (Chroma / HNSW)**: Eliminate slow brute-force $O(N)$ comparisons using approximate nearest neighbors indexing for lightning-fast sub-millisecond retrieval.